In [1]:
!pip install gensim

  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 591.3 kB/s  0:00:41m0:00:0100:02
Using cached smart_open-8.0.1-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gensim]2m2/3 [gensim]


In [4]:
import numpy as np
from gensim.models import Word2Vec

word2vec = Word2Vec.load("models/word2vec.model")

vector = word2vec.wv["жақсы"]
print("«жақсы» as a vector:")
print("  length     :", len(vector), "numbers")
print("  first five :", vector[:5].round(3))


vector = word2vec.wv["керемет"]
print("«керемет» as a vector:")
print("  length     :", len(vector), "numbers")
print("  first five :", vector[:5].round(3))

# Cosine similarity is only this much arithmetic:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("\nжақсы ~ керемет")
print("  by hand  :", round(cosine(word2vec.wv["жақсы"], word2vec.wv["керемет"]), 3))
print("  by gensim:", round(float(word2vec.wv.similarity("жақсы", "керемет")), 3))

# A single dimension on its own means nothing.
column = word2vec.wv.vectors[:, 7]
top = np.argsort(-column)[:5]
print("\nwords with the largest value in dimension 7:")
print(" ", ", ".join(word2vec.wv.index_to_key[i] for i in top))

«жақсы» as a vector:
  length     : 100 numbers
  first five : [-0.628 -0.191 -0.136  0.647 -0.079]
«керемет» as a vector:
  length     : 100 numbers
  first five : [-0.308 -0.161 -0.116  0.153 -0.226]

жақсы ~ керемет
  by hand  : 0.726
  by gensim: 0.726

words with the largest value in dimension 7:
  менің, ішінде, емес, соны, шығар


In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from dotenv import load_dotenv

load_dotenv(".env")
token = os.getenv("HF_TOKEN")

reviews = load_dataset("issai/kazsandra", "polarity_classification",
                       split="train", token=token).to_pandas()
balanced = pd.concat([g.sample(4000, random_state=42)      # the split of Lecture 3
                      for _, g in reviews.groupby("label")])

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
matrix = tfidf.fit_transform(balanced["text_cleaned"].astype(str))
cells = matrix.shape[0] * matrix.shape[1]

print("TF-IDF, as in Lecture 3:")
print("  matrix  :", matrix.shape[0], "documents ×", matrix.shape[1], "features")
print("  cells   :", f"{cells:,}")
print("  non-zero:", f"{matrix.nnz:,}", f"({100 * matrix.nnz / cells:.2f}%)")

print("\nembedding table, this lecture:")
print("  table   :", len(word2vec.wv), "words ×", word2vec.wv.vector_size, "dimensions")
print("  numbers :", f"{len(word2vec.wv) * word2vec.wv.vector_size:,}")
print("  non-zero: all of them")

TF-IDF, as in Lecture 3:
  matrix  : 8000 documents × 9419 features
  cells   : 75,352,000
  non-zero: 70,698 (0.09%)

embedding table, this lecture:
  table   : 27076 words × 100 dimensions
  numbers : 2,707,600
  non-zero: all of them


In [6]:
import re

TOKEN = re.compile(r"\w+")

def tokenize(text):
    return TOKEN.findall(str(text).lower())

corpus = [tokenize(text) for text in reviews["text_cleaned"].astype(str)]

print("reviews  :", len(corpus))
print("tokens   :", sum(len(sentence) for sentence in corpus))
print("longest  :", max(len(sentence) for sentence in corpus), "tokens")
print("distinct :", len(set(word for sentence in corpus for word in sentence)), "words")
print("example  :", corpus[0])

reviews  : 134368
tokens   : 1167795
longest  : 271 tokens
distinct : 119771 words
example  : ['өтте', 'күшті']


In [8]:
word2vec = Word2Vec(
    corpus,
    vector_size=100,   # numbers per word
    window=5,          # words of context on each side
    min_count=1,       # ignore words seen fewer than three times
    sg=1,              # 1 = skip-gram, 0 = CBOW
    epochs=10,         # passes over the corpus
    workers=1,         # one thread, so two runs give the same vectors
    seed=42,
)

os.makedirs("models", exist_ok=True)
word2vec.save("models/word2vec_kazsandra.model")

print("vocabulary :", len(word2vec.wv), "words")
print("per word   :", word2vec.wv.vector_size, "numbers")
print("dropped    :", len(set(w for s in corpus for w in s)) - len(word2vec.wv),
      "words seen fewer than once")
print("saved to   : models/word2vec_kazsandra.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary : 119771 words
per word   : 100 numbers
dropped    : 0 words seen fewer than once
saved to   : models/word2vec_kazsandra.model


In [9]:
word2vec_kazsandra = Word2Vec.load("models/word2vec_kazsandra.model")

for word in ("жақсы", "нашар", "интернет", "рахмет"):
    neighbours = word2vec_kazsandra.wv.most_similar(word, topn=5)
    print(word)
    print("  ", ", ".join(f"{w} {score:.2f}" for w, score in neighbours))

жақсы
   өте 0.78, женіл 0.75, керемет 0.75, жөргектің 0.75, күшті 0.75
нашар
   төмен 0.72, жаман 0.69, ұнамады 0.63, баяу 0.62, фуу 0.61
интернет
   сеть 0.69, актив 0.68, теле2 0.66, 4g 0.65, билаин 0.65
рахмет
   рақмет 0.89, рахмеет 0.83, ракмет 0.82, рахметт 0.78, алғысым 0.78


In [10]:
for a, b in (("жақсы", "керемет"), ("жақсы", "нашар"),
             ("интернет", "байланыс"), ("интернет", "тамақ")):
    print(f"{a:>9} ~ {b:<10} {word2vec_kazsandra.wv.similarity(a, b):.3f}")

    жақсы ~ керемет    0.753
    жақсы ~ нашар      0.403
 интернет ~ байланыс   0.609
 интернет ~ тамақ      0.303


In [11]:
for positive, negative in ((["әйел", "патша"], ["ер"]), (["жаман"], ["жақсы"])):
    found = word2vec_kazsandra.wv.most_similar(positive=positive, negative=negative, topn=3)
    print(f"{positive} - {negative} → {[w for w, _ in found]}")

['әйел', 'патша'] - ['ер'] → ['борыштық', 'арыздар', 'поэмалар']
['жаман'] - ['жақсы'] → ['косылайык', 'фуу', 'oqı']


In [12]:
word2vec_kazsandra = Word2Vec.load("models/word2vec_kazsandra.model")

review = "қосымша өте жақсы рахмет"
words = re.findall(r"\w+", review.lower())

vectors = [word2vec_kazsandra.wv[word] for word in words if word in word2vec_kazsandra.wv]
document = np.mean(vectors, axis=0)

print("review   :", review)
print("words    :", words)
print("each word:", vectors[0].shape, "→", len(vectors), "vectors")
print("document :", document.shape)
print("first few:", document[:5].round(3))

review   : қосымша өте жақсы рахмет
words    : ['қосымша', 'өте', 'жақсы', 'рахмет']
each word: (100,) → 4 vectors
document : (100,)
first few: [-0.282 -0.037 -0.115  0.308 -0.399]


In [13]:
from gensim.models import FastText
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# 1. the models trained earlier — nothing is retrained here
word2vec = Word2Vec.load("models/word2vec.model")
fasttext = FastText.load("models/fasttext.model")
word2vec_kazsandra = Word2Vec.load("models/word2vec_kazsandra.model")


# 2. the drop-in replacement for TfidfVectorizer
class MeanEmbedding(BaseEstimator, TransformerMixin):
    """Average the word vectors of a document."""

    def __init__(self, model=None):
        self.model = model

    def fit(self, x, y=None):
        return self                       # nothing to learn: the vectors exist

    def transform(self, texts):
        vectors = self.model.wv
        out = np.zeros((len(texts), vectors.vector_size), dtype=np.float32)
        self.skipped_ = 0
        for i, text in enumerate(texts):
            found = []
            for word in re.findall(r"\w+", str(text).lower()):
                try:
                    found.append(vectors[word])
                except KeyError:          # word2vec only: unknown word
                    self.skipped_ += 1
            if found:
                out[i] = np.mean(found, axis=0)
        return out


# 3. the balanced split of Lecture 3
def balance(df, per_class, seed=42):
    parts = [g.sample(per_class, random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def split(name, per_class):
    df = load_dataset("issai/kazsandra", "polarity_classification",
                      split=name, token=token).to_pandas()
    df = balance(df, per_class)
    return df["text_cleaned"].astype(str), df["label"]

x_train, y_train = split("train", 4000)
x_test, y_test = split("test", 1000)

# 4. the same classifier, three kinds of features
classifier = lambda: LogisticRegression(max_iter=2000, C=5.0)
models = {
    "TF-IDF": Pipeline([("features", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                                     sublinear_tf=True)),
                        ("clf", classifier())]),
    "word2vec": Pipeline([("features", MeanEmbedding(word2vec)), ("clf", classifier())]),
    "fastText": Pipeline([("features", MeanEmbedding(fasttext)), ("clf", classifier())]),
    "word2vec_kazsandra": Pipeline([("features", MeanEmbedding(word2vec_kazsandra)), ("clf", classifier())]),
}

predictions = {}
for name, model in models.items():
    model.fit(x_train, y_train)
    predictions[name] = model.predict(x_test)
    scores = cross_val_score(model, x_train, y_train, cv=5)
    skipped = getattr(model.named_steps["features"], "skipped_", 0)
    print(f"{name:<10} test={model.score(x_test, y_test):.3f}"
          f"   CV={scores.mean():.3f} ± {scores.std():.3f}"
          f"   skipped {skipped} unknown tokens")

print()
print(classification_report(y_test, predictions["fastText"],
                            target_names=["negative", "positive"], digits=3))
print("confusion matrix (rows = true, columns = predicted)")
print(confusion_matrix(y_test, predictions["fastText"]))

TF-IDF     test=0.780   CV=0.780 ± 0.006   skipped 0 unknown tokens
word2vec   test=0.794   CV=0.795 ± 0.008   skipped 2156 unknown tokens
fastText   test=0.792   CV=0.800 ± 0.006   skipped 0 unknown tokens
word2vec_kazsandra test=0.797   CV=0.795 ± 0.011   skipped 1380 unknown tokens

              precision    recall  f1-score   support

    negative      0.807     0.769     0.788      1000
    positive      0.779     0.816     0.797      1000

    accuracy                          0.792      2000
   macro avg      0.793     0.792     0.792      2000
weighted avg      0.793     0.792     0.792      2000

confusion matrix (rows = true, columns = predicted)
[[769 231]
 [184 816]]


In [14]:
names = {0: "negative", 1: "positive"}
rows = list(zip(x_test, y_test, predictions["TF-IDF"], predictions["fastText"]))
fixed = [(t, y) for t, y, a, b in rows if a != y and b == y]
broken = [(t, y) for t, y, a, b in rows if a == y and b != y]

print(f"\nfastText fixed {len(fixed)} reviews that TF-IDF got wrong")
print(f"fastText broke {len(broken)} reviews that TF-IDF got right")
print(f"net change on {len(rows)} reviews: {len(fixed) - len(broken):+d}")
for text, label in fixed[:2]:
    print(f"  fixed  {names[label]:>8}: «{text[:46]}»")
for text, label in broken[:2]:
    print(f"  broke  {names[label]:>8}: «{text[:46]}»")


fastText fixed 164 reviews that TF-IDF got wrong
fastText broke 140 reviews that TF-IDF got right
net change on 2000 reviews: +24
  fixed  positive: «ма шааллагь»
  fixed  positive: «жаңа әндер мен өлеңдер жинағы мен сені чаттан »
  broke  negative: «кемшілігі көп»
  broke  positive: «өте керемет екен дартын дар өкімписін дер»
